# Módulo 38 · Proyecto C: de nombres de ligandos a un archivo SDF

Partimos de la lista de heteromoléculas del curso (código PDB de 3 letras), consultamos su SMILES en la API del PDB, construimos las moléculas con RDKit, calculamos su peso molecular y guardamos todo en un SDF. **Requiere conexión.**

In [ ]:
# Instalación (solo la primera vez en Colab; en tu ordenador: pip install rdkit seaborn pandas matplotlib)
!pip -q install rdkit seaborn


Archivos de datos del curso (se crean en la carpeta de trabajo):

In [ ]:
# Archivos de datos del curso
open("heteromoleculas.csv", "w").write("Nombre,ID de tres letras\nHEME C,HEC\nUNDECYL-MALTOSIDE,UMQ\n(1R)-2-(dodecanoyloxy)-1-[(phosphonooxy)methyl]ethyl tetradecanoate,7PH\nOctadecane,8K6\n\"(1S,8E)-1-{[(2S)-3-hydroxy-2-{[(1S)-1-hydroxyoctadecyl]oxy}propyl]oxy}octadec-8-en-1-ol\",2WM\nCHLOROPHYLL A,CLA\n\"(7R,17E)-4-HYDROXY-N,N,N,7-TETRAMETHYL-7-[(8E)-OCTADEC-8-ENOYLOXY]-10-OXO-3,5,9-TRIOXA-4-PHOSPHAHEPTACOS-17-EN-1-AMINIUM 4-OXIDE\",OPC\nCADMIUM ION,CD\nPENTADECANE,MYS\n\"1,2-DI-O-ACYL-3-O-[6-DEOXY-6-SULFO-ALPHA-D-GLUCOPYRANOSYL]-SN-GLYCEROL\",SQD\nFE2/S2 (INORGANIC) CLUSTER,FES\n(1R)-1-{[(2S)-3-hydroxy-2-{[(1R)-1-hydroxypentyl]oxy}propyl]oxy}hexan-1-ol,2WD\n\"(1S,8E,1'R,8'Z)-1,1'-{[(2S)-3-hydroxypropane-1,2-diyl]bis(oxy)}bisoctadec-8-en-1-ol\",3WM\n\"(1S,8E)-1-{[(2S)-1-hydroxy-3-{[(1S)-1-hydroxypentadecyl]oxy}propan-2-yl]oxy}heptadec-8-en-1-ol\",2WA\nN-OCTANE,OCT\n(2S)-3-(alpha-D-galactopyranosyloxy)-2-(hexadecanoyloxy)propyl (9Z)-octadec-9-enoate,1O2\nBETA-CAROTENE,BCR\nwater,HOH\n")
print("Archivos creados:", ["heteromoleculas.csv"])

In [ ]:
import time
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

het = pd.read_csv("heteromoleculas.csv")
print(het.head())

def smiles_pdb(codigo):
    r = requests.get(f"https://data.rcsb.org/rest/v1/core/chemcomp/{codigo}", timeout=20)
    if r.status_code != 200:
        return None
    return r.json().get("rcsb_chem_comp_descriptor", {}).get("smiles")

smiles = []
for c in het["ID de tres letras"]:
    smiles.append(smiles_pdb(c))
    time.sleep(0.2)                       # cortesía con el servidor
het["SMILES"] = smiles
het["mol"] = het["SMILES"].apply(lambda s: Chem.MolFromSmiles(s) if s else None)
het["peso"] = het["mol"].apply(lambda m: round(Descriptors.MolWt(m), 1) if m else None)
print(het[["ID de tres letras", "peso", "SMILES"]].sort_values("peso", ascending=False).head(10))

writer = Chem.SDWriter("heteromoleculas.sdf")
for _, fila in het.dropna(subset=["peso"]).iterrows():
    fila["mol"].SetProp("_Name", fila["ID de tres letras"])
    fila["mol"].SetProp("MW", str(fila["peso"]))
    writer.write(fila["mol"])
writer.close()
print("Moléculas guardadas:", len(Chem.SDMolSupplier("heteromoleculas.sdf")))